In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

# Шестёрка выбрана жадным отбором на честном фолде: 0.6627 против 0.6576 у четырёх.
# Седьмой давал +0.0002 и отброшен.
# Четвёрка на одной основе: они делят токенизатор, значит на инференсе одна токенизация
# на все четыре и четыре прохода вместо шести. Шестёрка давала 0.6753, но трижды не
# уложилась во время.
ENCODERS = ("ce_relaxed", "ce_combo", "ce_spec", "ce_self")
fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
# Пути разрешаются по файлу-опознавателю, который есть только в нужном каталоге:
# `ce_relaxed.npy` лежит в трёх местах сразу, и маска по всем входам берёт не тот.
#   ce_balanced.npy   — только в датасете скоров на фолде (там все десять энкодеров)
#   clean_pairs.parquet       — только в выходе первого прохода (четыре энкодера)
#   clean_pairs_check.parquet — только в выходе второго прохода (ce_e5 и ce_ru)
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_balanced.npy", recursive=True)[0])
extra = os.path.dirname(glob.glob("/kaggle/input/**/clean_pairs.parquet", recursive=True)[0])
extra2 = os.path.dirname(glob.glob("/kaggle/input/**/clean_pairs_check.parquet", recursive=True)[0])
log(f"скоры фолда: {sc}")
log(f"первый проход: {extra}")
log(f"второй проход: {extra2}")
for e in ENCODERS:
    assert os.path.exists(f"{sc}/{e}.npy"), f"нет скоров фолда для {e}"
    assert os.path.exists(f"{extra}/{e}.npy") or os.path.exists(f"{extra2}/{e}.npy"), \
        f"нет скоров новых пар для {e}"
log("все шесть наборов скоров найдены с обеих сторон")

def prepare(pairs, items, scores, tag):
    """Признаки, величины и структурная модель для набора пар."""
    y = (pairs["target"].to_numpy() > 0).astype(np.int8) if "target" in pairs else \
        (pairs["label"].to_numpy() > 0).astype(np.int8)
    cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
    names = list(feature_names(False, True))
    X = np.zeros((len(pairs), len(names)), dtype=np.float32)
    for c in sorted(set(items.category.astype(str))):
        rows = np.flatnonzero(cat == c)
        if not len(rows): continue
        sub = items[items.category.astype(str) == c].reset_index(drop=True)
        X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), KNOWN,
                               with_neighbours=False, with_measures=True)
        del sub; gc.collect()
    legacy = extract_model_features(pairs[["id1", "id2"]], items)
    pr = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
    au = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
    del legacy; gc.collect()
    codes = np.array([KNOWN.index(c) if c in KNOWN else -1 for c in cat], dtype=np.float32)
    M = np.column_stack([X] + [scores[e] for e in ENCODERS] + [0.8*pr + 0.2*au, codes])
    log(f"  {tag}: {M.shape}, доля+ {y.mean():.3f}")
    return M.astype(np.float32), y, cat

vp = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
vi = pd.read_parquet(fold + "/llm_valid_items.parquet")
KNOWN = sorted(set(vi.category.astype(str)))
Mv, yv, catv = prepare(vp, vi, {e: np.load(f"{sc}/{e}.npy") for e in ENCODERS}, "фолд")

ep = pd.read_parquet(extra + "/clean_pairs.parquet")
ei = pd.read_parquet(extra + "/clean_items.parquet")
# Два прохода отбирали пары независимо, одним и тем же зерном. Сверяем поимённо: если
# порядок разъедется, скоры четырёх и двух энкодеров нельзя складывать в одну матрицу.
check = pd.read_parquet(extra2 + "/clean_pairs_check.parquet")
same = (len(check) == len(ep) and np.array_equal(check.id1.to_numpy(), ep.id1.to_numpy())
        and np.array_equal(check.id2.to_numpy(), ep.id2.to_numpy()))
log(f"пары двух проходов совпали: {same}")
if not same:
    raise SystemExit("отбор пар разъехался между проходами — складывать скоры нельзя")
scores_extra = {}
for e in ENCODERS:
    path = f"{extra}/{e}.npy" if os.path.exists(f"{extra}/{e}.npy") else f"{extra2}/{e}.npy"
    scores_extra[e] = np.load(path)
Me, ye, cate = prepare(ep, ei, scores_extra, "новые пары")

masks = {c: catv == c for c in np.unique(catv)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[yv[rows]==1], rows[yv[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(yv[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(yv)) % 2
# Честно: половина фолда для замера никогда не входит в обучение, новые пары входят целиком.
oof_small = np.zeros(len(yv)); oof_big = np.zeros(len(yv))
for h in (0, 1):
    tr, te = half != h, half == h
    oof_small[te] = HistGradientBoostingClassifier(**PARAMS).fit(Mv[tr], yv[tr]).predict_proba(Mv[te])[:, 1]
    big = HistGradientBoostingClassifier(**PARAMS).fit(np.vstack([Mv[tr], Me]), np.concatenate([yv[tr], ye]))
    oof_big[te] = big.predict_proba(Mv[te])[:, 1]
a, sa = macro(rk(oof_small)); b, sb = macro(rk(oof_big))
log(f"только фолд ({len(yv):,} пар):        {a:.6f} ± {sa:.6f}")
log(f"фолд + новые ({len(yv)+len(ye):,} пар): {b:.6f} ± {sb:.6f}   ({b-a:+.6f})")

use_extra = b > a
Xall = np.vstack([Mv, Me]) if use_extra else Mv
yall = np.concatenate([yv, ye]) if use_extra else yv
final = HistGradientBoostingClassifier(**PARAMS).fit(Xall, yall)
trees = export(final)
save("/kaggle/working/fusion_boost.npz", trees)
ok = np.allclose(predict_proba(trees, Xall[:2000]), final.predict_proba(Xall[:2000])[:, 1], atol=1e-6)
COLS = list(feature_names(False, True)) + list(ENCODERS) + ["structural", "category_code"]
json.dump({"columns": COLS, "categories": KNOWN, "params": PARAMS,
           "honest_macro": max(a, b), "n_train": int(len(yall)),
           "uses_structural": True, "encoders": list(ENCODERS),
           "used_extra_pairs": bool(use_extra)},
          open("/kaggle/working/fusion_info.json", "w"), ensure_ascii=False, indent=1)
log(f"выгружено: {len(COLS)} столбцов, обучено на {len(yall):,} парах, совпадение {ok}")
